# AntibodySteering Colab Workflow

This notebook keeps the workflow as close as possible to the command-line version of this repo.

Important constraint: Colab GPU runs on a remote Google VM, not on your Mac. That means it cannot directly read files from Finder. To use Colab GPUs, your files have to reach the Colab runtime somehow:
- clone the repo from GitHub
- upload files into the runtime
- or mount Google Drive

The code paths and commands stay the same. The main difference is that paths point to `/content/...` instead of your local filesystem.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path


def run(cmd: str, check: bool = True) -> None:
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=check)


IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)


Running in Colab: True


## 1. Get the repo into the Colab runtime

Use GitHub if you want the closest match to your local CLI workflow.

In [3]:
REPO_URL = "https://github.com/MaxWZhuang/steerable-antibody-gen.git"
REPO_BRANCH = "main"
PROJECT_DIR = Path("/content/AntibodySteering")

if PROJECT_DIR.exists():
    print(f"Using existing repo at {PROJECT_DIR}")
else:
    if "YOUR_GITHUB_USERNAME" in REPO_URL:
        raise ValueError("Update REPO_URL before running this cell.")
    run(shlex.join(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)]))

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())


$ git clone --branch main https://github.com/MaxWZhuang/steerable-antibody-gen.git /content/AntibodySteering
cwd: /content/AntibodySteering


## 2. Install dependencies and verify GPU

In [4]:
run("pip install -q -e .")
run("pip install -q pandas tabulate tqdm")

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    run("nvidia-smi")


$ pip install -q -e .
$ pip install -q pandas tabulate tqdm
torch: 2.10.0+cpu
cuda available: False


## 3. Choose where your data lives inside Colab

Default is plain runtime storage under `/content`, which is the closest to local CLI usage.

Use Drive only if you want persistence across Colab restarts.

In [5]:
USE_DRIVE = False

if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Set USE_DRIVE = False if you are not in Colab.")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DATA_ROOT = Path("/content/drive/MyDrive/antibody_colab")
else:
    DATA_ROOT = Path("/content/antibody_colab")

RAW_OAS_DIR = DATA_ROOT / "oas_raw"
PROCESSED_DIR = DATA_ROOT / "prepared_oas"
CHECKPOINT_ROOT = DATA_ROOT / "checkpoints"

RAW_OAS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("RAW_OAS_DIR:", RAW_OAS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)


DATA_ROOT: /content/antibody_colab
RAW_OAS_DIR: /content/antibody_colab/oas_raw
PROCESSED_DIR: /content/antibody_colab/prepared_oas
CHECKPOINT_ROOT: /content/antibody_colab/checkpoints


## 4. Optional: upload files from your laptop into the Colab runtime

This is the closest thing to dragging files from Finder into Colab, but it is still an upload into the remote VM.

Use this for smaller files. For large datasets, GitHub, Drive, or another cloud source is usually better.

Set `UPLOAD_TARGET = "processed"` to upload prebuilt `.jsonl.gz` files into `PROCESSED_DIR`.

In [11]:
UPLOAD_FILES = True
UPLOAD_TARGET = "processed"  # choose: "raw" or "processed"

target_dir = RAW_OAS_DIR if UPLOAD_TARGET == "raw" else PROCESSED_DIR
print("upload target:", target_dir)

if UPLOAD_FILES:
    if not IN_COLAB:
        raise RuntimeError("File upload helper only works in Colab.")
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = target_dir / name
        target.write_bytes(content)
        print(f"Saved {name} -> {target}")
else:
    print("Set UPLOAD_FILES = True if you want to upload files from your computer.")
    print("For processed data, leave UPLOAD_TARGET = 'processed' and upload files like oas_all.jsonl.gz or oas_paired.jsonl.gz.")


upload target: /content/antibody_colab/prepared_oas


KeyboardInterrupt: 

## 5. Optional: preprocess raw OAS data

This uses the same script and flags as your local command line.

In [ ]:
PREPARE_DATA = False

INPUT_DIR = RAW_OAS_DIR
OUTPUT_DIR = PROCESSED_DIR
STATS_PATH = DATA_ROOT / "prepare_stats.json"

MAX_FILES = 100
MAX_RECORDS = None
VAL_PERCENT = 10
SAMPLING_MODE = "round_robin"
CHAIN_BALANCE_ALPHA = 0.0
REQUIRE_COMPLETE_VDJ = False

prepare_cmd = [
    "python",
    "scripts/prepare_oas.py",
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--stats-output",
    str(STATS_PATH),
    "--max-files",
    str(MAX_FILES),
    "--val-percent",
    str(VAL_PERCENT),
    "--sampling-mode",
    SAMPLING_MODE,
    "--chain-balance-alpha",
    str(CHAIN_BALANCE_ALPHA),
]

if MAX_RECORDS is not None:
    prepare_cmd.extend(["--max-records", str(MAX_RECORDS)])

if REQUIRE_COMPLETE_VDJ:
    prepare_cmd.append("--require-complete-vdj")

print(shlex.join(prepare_cmd))
if PREPARE_DATA:
    run(shlex.join(prepare_cmd))
else:
    print("Set PREPARE_DATA = True to run preprocessing.")


## 6. Write the training config

Same config shape, same training script, same flags. Only the filesystem root changes.

In [9]:
import yaml

TRAINING_STAGE = "paired_refine"

BASE_DATA_PATH = PROCESSED_DIR / "oas_all.jsonl.gz"
PAIRED_DATA_PATH = PROCESSED_DIR / "oas_paired.jsonl.gz"
BASE_CHECKPOINT_DIR = CHECKPOINT_ROOT / "mlm_oas_base"
PAIRED_CHECKPOINT_DIR = CHECKPOINT_ROOT / "mlm_oas_paired_refine"
INIT_CHECKPOINT = BASE_CHECKPOINT_DIR / "best.pt"

config = {
    "data_path": str(BASE_DATA_PATH),
    "output_dir": str(BASE_CHECKPOINT_DIR),
    "batch_size": 64,
    "eval_batch_size": 64,
    "train_num_workers": 2,
    "eval_num_workers": 2,
    "max_length": 192,
    "bucket_width": 8,
    "mask_probability": 0.15,
    "hcdr3_span_probability": 0.4,
    "hcdr3_span_min": 3,
    "hcdr3_span_max": 8,
    "shuffle_pair_probability": 0.0,
    "pair_loss_weight": 0.0,
    "d_model": 256,
    "n_heads": 8,
    "n_layers": 6,
    "d_ff": 1024,
    "dropout": 0.1,
    "learning_rate": 3e-4,
    "weight_decay": 1e-2,
    "grad_clip_norm": 1.0,
    "epochs": 5,
    "seed": 42,
    "use_amp": True,
    "show_progress": True,
}

if TRAINING_STAGE == "paired_refine":
    config.update(
        {
            "training_stage": "paired_refine",
            "data_path": str(PAIRED_DATA_PATH),
            "output_dir": str(PAIRED_CHECKPOINT_DIR),
            "init_checkpoint": str(INIT_CHECKPOINT),
            "batch_size": 32,
            "eval_batch_size": 32,
            "shuffle_pair_probability": 0.5,
            "pair_loss_weight": 0.5,
            "learning_rate": 1e-4,
        }
    )

config_path = Path("configs/colab_train.yaml")
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(config_path.read_text(encoding="utf-8"))


data_path: /content/antibody_colab/prepared_oas/oas_paired.jsonl.gz
output_dir: /content/antibody_colab/checkpoints/mlm_oas_paired_refine
batch_size: 32
eval_batch_size: 32
train_num_workers: 2
eval_num_workers: 2
max_length: 192
bucket_width: 8
mask_probability: 0.15
hcdr3_span_probability: 0.4
hcdr3_span_min: 3
hcdr3_span_max: 8
shuffle_pair_probability: 0.5
pair_loss_weight: 0.5
d_model: 256
n_heads: 8
n_layers: 6
d_ff: 1024
dropout: 0.1
learning_rate: 0.0001
weight_decay: 0.01
grad_clip_norm: 1.0
epochs: 5
seed: 42
use_amp: true
show_progress: true
training_stage: paired_refine
init_checkpoint: /content/antibody_colab/checkpoints/mlm_oas_base/best.pt



## 7. Optional smoke test

In [ ]:
RUN_SMOKE_TEST = False

smoke_cmd = [
    "python",
    "scripts/mlm_train.py",
    "--config",
    "configs/colab_train.yaml",
    "--smoke-test-only",
]

print(shlex.join(smoke_cmd))
if RUN_SMOKE_TEST:
    run(shlex.join(smoke_cmd))


## 8. Train

This is the same training entrypoint you use locally.

In [10]:
TRAIN = True

train_cmd = [
    "python",
    "scripts/mlm_train.py",
    "--config",
    "configs/colab_train.yaml",
]

print(shlex.join(train_cmd))
if TRAIN:
    run(shlex.join(train_cmd))


python scripts/mlm_train.py --config configs/colab_train.yaml
$ python scripts/mlm_train.py --config configs/colab_train.yaml


CalledProcessError: Command 'python scripts/mlm_train.py --config configs/colab_train.yaml' returned non-zero exit status 1.

## 9. Inspect outputs

In [ ]:
run_dir = Path(config["output_dir"])
print("run dir:", run_dir)

if run_dir.exists():
    for path in sorted(run_dir.iterdir()):
        print(path.name, path.stat().st_size, "bytes")

train_config_json = run_dir / "train_config.json"
if train_config_json.exists():
    print("\ntrain_config.json")
    print(train_config_json.read_text(encoding="utf-8"))


## 10. Download checkpoints to your Mac

This is the simplest local-storage workflow for Colab GPU runs: train in `/content`, zip the checkpoint folder, then download it through the browser.

In [ ]:
DOWNLOAD_CHECKPOINTS = False

zip_path = Path("/content") / f"{run_dir.name}.zip"
zip_cmd = f"zip -r {shlex.quote(str(zip_path))} {shlex.quote(str(run_dir))}"
print(zip_cmd)

if DOWNLOAD_CHECKPOINTS:
    if not IN_COLAB:
        raise RuntimeError("Checkpoint download helper only works in Colab.")
    if not run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_dir}")
    run(zip_cmd)
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Set DOWNLOAD_CHECKPOINTS = True after training to download a zip of the checkpoint folder.")


## 11. Run arbitrary CLI commands

Anything you normally type into the terminal can go here too.

In [ ]:
EXTRA_COMMAND = "python scripts/inspect_oas.py /content/path/to/file.csv.gz"
print(EXTRA_COMMAND)
# run(EXTRA_COMMAND)
